# Módulo 4 · Clase 5 — Python + SQL

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema | Herramienta |
|---|------|-------------|
| 1 | Conexión y cursor | `sqlite3` (stdlib) |
| 2 | Ejecutar sentencias | `execute`, `executemany`, `executescript` |
| 3 | Recuperar resultados | `fetchone`, `fetchall`, `fetchmany` |
| 4 | Parámetros seguros | `?` — prevenir SQL injection |
| 5 | Context manager | `with conn:` para transacciones |
| 6 | Integración con pandas | `read_sql_query`, `to_sql` |
| 7 | SQLAlchemy — Core | Engine, conexión, texto SQL |
| 8 | SQLAlchemy — ORM | Modelos, sesión, CRUD |
| 9 | Práctica | Pipeline de análisis completo |

---
## 1. `sqlite3` — el módulo de la biblioteca estándar

No necesitas instalar nada. `sqlite3` viene incluido con Python.

```python
import sqlite3

# Archivo en disco
conn = sqlite3.connect("mi_base.db")

# Base de datos en memoria (se pierde al cerrar)
conn = sqlite3.connect(":memory:")

# Crear cursor
cur = conn.cursor()

# Ejecutar
cur.execute("SELECT ...")

# Confirmar cambios
conn.commit()

# Cerrar
conn.close()
```

In [ ]:
import sqlite3
from pathlib import Path

DB = Path.home() / "demo_python_sql.db"
conn = sqlite3.connect(DB)
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

# Crear tabla y poblar
cur.executescript("""
    DROP TABLE IF EXISTS tareas;
    CREATE TABLE tareas (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        titulo      TEXT    NOT NULL,
        completada  INTEGER DEFAULT 0,
        prioridad   TEXT    DEFAULT 'normal'
    );
""")

cur.executemany(
    "INSERT INTO tareas (titulo, prioridad) VALUES (?, ?)",
    [
        ("Revisar correo",         "normal"),
        ("Preparar informe",       "alta"),
        ("Llamar al cliente",      "alta"),
        ("Actualizar documentación","baja"),
        ("Sprint planning",        "alta"),
        ("Code review",            "normal"),
    ]
)
conn.commit()
print(f"Base de datos creada en {DB}")

---
## 2 y 3. Ejecutar y recuperar resultados

```python
cur.execute("SELECT ...")

cur.fetchone()          # → tuple con la primera fila, o None
cur.fetchall()          # → list[tuple] con todas las filas
cur.fetchmany(n)        # → list[tuple] con las siguientes n filas

# Iterar directamente sobre el cursor
for fila in cur.execute("SELECT ..."):
    print(fila)

# Acceso por nombre de columna (en lugar de por índice)
conn.row_factory = sqlite3.Row
cur.execute("SELECT nombre, precio FROM productos")
fila = cur.fetchone()
print(fila['nombre'], fila['precio'])
```

In [ ]:
# fetchone
cur.execute("SELECT * FROM tareas WHERE prioridad = 'alta'")
primera = cur.fetchone()
print("fetchone:", primera)

# fetchall
cur.execute("SELECT id, titulo FROM tareas ORDER BY id")
todas = cur.fetchall()
print("\nfetchall:", todas)

# Iterar sobre el cursor
print("\nIterar sobre cursor:")
for id_, titulo, completada, prioridad in cur.execute("SELECT * FROM tareas ORDER BY prioridad"):
    estado = "✅" if completada else "⬜"
    print(f"  {estado} [{prioridad:6s}] {titulo}")

# Acceso por nombre con row_factory
conn.row_factory = sqlite3.Row
cur = conn.cursor()
cur.execute("SELECT * FROM tareas LIMIT 2")
print("\nrow_factory — acceso por nombre:")
for fila in cur.fetchall():
    print(f"  id={fila['id']} titulo='{fila['titulo']}' prioridad={fila['prioridad']}")

---
## 4. Parámetros seguros — prevenir SQL injection

**Nunca** construyas sentencias SQL con f-strings o concatenación de strings a partir de datos del usuario.

```python
# ❌ PELIGROSO — SQL injection
titulo = input("Título:")  # usuario escribe: ' OR 1=1 --
cur.execute(f"SELECT * FROM tareas WHERE titulo = '{titulo}'")
#           → SELECT * FROM tareas WHERE titulo = '' OR 1=1 --'
#             Devuelve TODAS las filas

# ✅ SEGURO — parámetro ?
cur.execute("SELECT * FROM tareas WHERE titulo = ?", (titulo,))
# sqlite3 escapa el valor automáticamente
```

**Regla de oro:** los datos van siempre como parámetros, nunca interpolados en el SQL.

In [ ]:
# Buscar por prioridad (parámetro seguro)
def buscar_por_prioridad(conn, prioridad):
    cur = conn.cursor()
    cur.execute("SELECT id, titulo FROM tareas WHERE prioridad = ?", (prioridad,))
    return cur.fetchall()

for p in ['alta', 'normal', 'baja']:
    resultado = buscar_por_prioridad(conn, p)
    print(f"\nPrioridad '{p}':")
    for fila in resultado:
        print(f"  [{fila[0]}] {fila[1]}")

# Insertar con parámetros
nueva = ("Desplegar a producción", "alta")
cur.execute("INSERT INTO tareas (titulo, prioridad) VALUES (?, ?)", nueva)
nuevo_id = cur.lastrowid
conn.commit()
print(f"\nNueva tarea insertada con id={nuevo_id}")

---
## 5. Context manager para transacciones

```python
# with conn actúa como bloque de transacción
with conn:
    conn.execute("UPDATE ...")
    conn.execute("INSERT ...")
# Si no hay excepción → commit automático
# Si hay excepción    → rollback automático
```

También puedes abrir y cerrar la conexión con `with`:

```python
with sqlite3.connect("mi_bd.db") as conn:
    conn.execute("INSERT ...")
# conn se cierra automáticamente al salir del bloque
```

In [ ]:
# Transacción exitosa
with conn:
    conn.execute("UPDATE tareas SET completada = 1 WHERE id = 1")
    conn.execute("UPDATE tareas SET completada = 1 WHERE id = 2")
print("Transacción completada")

# Transacción fallida → rollback
import contextlib

try:
    with conn:
        conn.execute("UPDATE tareas SET completada = 1 WHERE id = 3")
        raise ValueError("Algo salió mal")  # simular error
        conn.execute("UPDATE tareas SET completada = 1 WHERE id = 4")
except ValueError as e:
    print(f"Error capturado: {e} → rollback automático")

# Verificar estado
cur.execute("SELECT id, titulo, completada FROM tareas")
for fila in cur.fetchall():
    print(f"  id={fila[0]} completada={fila[2]} {fila[1]}")

---
## 6. Integración con pandas

```python
import pandas as pd

# Leer resultado SQL como DataFrame
df = pd.read_sql_query("SELECT * FROM tabla WHERE ...", conn)
df = pd.read_sql_query("SELECT * FROM tabla WHERE id = ?", conn, params=(5,))

# Escribir DataFrame a tabla SQL
df.to_sql(
    name='nombre_tabla',
    con=conn,
    if_exists='replace',  # 'replace' | 'append' | 'fail'
    index=False
)
```

In [ ]:
import pandas as pd

# Leer de SQLite a DataFrame
df_tareas = pd.read_sql_query(
    "SELECT * FROM tareas ORDER BY prioridad, id",
    conn
)
print("DataFrame desde SQLite:")
display(df_tareas)

# Análisis con pandas
print("\nResumen por prioridad:")
display(df_tareas.groupby('prioridad').agg(
    total=('id', 'count'),
    completadas=('completada', 'sum')
).reset_index())

# Escribir DataFrame a SQL
nuevas = pd.DataFrame([
    {'titulo': 'Revisar pull requests', 'prioridad': 'alta',   'completada': 0},
    {'titulo': 'Actualizar dependencias','prioridad': 'baja',   'completada': 0},
    {'titulo': 'Escribir tests',          'prioridad': 'normal','completada': 0},
])

nuevas.to_sql('tareas', conn, if_exists='append', index=False)
print(f"\nTotal tareas ahora: {pd.read_sql_query('SELECT COUNT(*) AS n FROM tareas', conn).iloc[0]['n']}")

---
## 7. SQLAlchemy — Core

SQLAlchemy es la librería de base de datos más usada en Python. Tiene dos capas:

```
SQLAlchemy ORM   ← nivel alto: Python objects ↔ tabla SQL
SQLAlchemy Core  ← nivel medio: SQL expresivo en Python
      ↓
DB API 2.0       ← nivel bajo: sqlite3, psycopg2, etc.
```

**Instalación** (si no la tienes):
```bash
pip install sqlalchemy
```

In [ ]:
try:
    import sqlalchemy
    print(f"SQLAlchemy {sqlalchemy.__version__} disponible")
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "sqlalchemy"], check=True)
    import sqlalchemy
    print(f"SQLAlchemy {sqlalchemy.__version__} instalado")

In [ ]:
from sqlalchemy import create_engine, text
from pathlib import Path

# El engine es la conexión al motor de BD
engine = create_engine(f"sqlite:///{Path.home() / 'tienda.db'}")

# Ejecutar SQL en texto con SQLAlchemy Core
with engine.connect() as con:
    # SELECT
    resultado = con.execute(text("SELECT nombre, precio FROM productos ORDER BY precio DESC LIMIT 5"))
    print("Top 5 productos:")
    for fila in resultado:
        print(f"  {fila.nombre:30s} {fila.precio:>8.2f}€")

    # SELECT con parámetros (usando :nombre)
    resultado = con.execute(
        text("SELECT nombre, stock FROM productos WHERE categoria_id = :cat"),
        {"cat": 2}
    )
    print("\nProductos de Informática:")
    for fila in resultado:
        print(f"  {fila.nombre:30s} stock={fila.stock}")

---
## 8. SQLAlchemy — ORM

El ORM (Object-Relational Mapper) mapea tablas SQL a clases Python. Cada fila se convierte en un objeto.

```python
# Tabla SQL            ↔     Clase Python
# Fila                 ↔     Instancia
# Columna              ↔     Atributo
# INSERT               ↔     session.add(objeto)
# SELECT               ↔     session.query(Clase).filter(...)
# UPDATE               ↔     objeto.atributo = nuevo_valor
# DELETE               ↔     session.delete(objeto)
# COMMIT               ↔     session.commit()
```

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, Session

# Motor (base de datos en memoria para este demo)
engine_orm = create_engine("sqlite:///:memory:", echo=False)
Base = declarative_base()

# Definir modelos (clases = tablas)
class Categoria(Base):
    __tablename__ = "categorias"
    id       = Column(Integer, primary_key=True)
    nombre   = Column(String, nullable=False)
    productos = relationship("Producto", back_populates="categoria")

    def __repr__(self):
        return f"<Categoria id={self.id} nombre='{self.nombre}'>"

class Producto(Base):
    __tablename__ = "productos"
    id           = Column(Integer, primary_key=True)
    nombre       = Column(String, nullable=False)
    precio       = Column(Float, nullable=False)
    stock        = Column(Integer, default=0)
    categoria_id = Column(Integer, ForeignKey("categorias.id"))
    categoria    = relationship("Categoria", back_populates="productos")

    def __repr__(self):
        return f"<Producto '{self.nombre}' {self.precio}€>"

# Crear las tablas
Base.metadata.create_all(engine_orm)

# CRUD con sesión
with Session(engine_orm) as session:
    # CREATE
    electronica = Categoria(nombre="Electrónica")
    informatica = Categoria(nombre="Informática")
    session.add_all([electronica, informatica])
    session.flush()  # asigna IDs sin hacer commit todavía

    productos = [
        Producto(nombre="Smartphone",  precio=599.99, stock=50, categoria=electronica),
        Producto(nombre="Auriculares", precio=89.99,  stock=120, categoria=electronica),
        Producto(nombre="Portátil",    precio=1199.0, stock=30,  categoria=informatica),
        Producto(nombre="Teclado",     precio=85.0,   stock=75,  categoria=informatica),
    ]
    session.add_all(productos)
    session.commit()

    # READ — query básica
    todos = session.query(Producto).all()
    print("Todos los productos:")
    for p in todos:
        print(f"  {p.nombre:15s} {p.precio:>8.2f}€  cat={p.categoria.nombre}")

    # READ — filtro
    caros = session.query(Producto).filter(Producto.precio > 100).order_by(Producto.precio.desc()).all()
    print("\nProductos > 100€:", caros)

    # UPDATE
    smartphone = session.query(Producto).filter_by(nombre="Smartphone").first()
    smartphone.precio = 549.99
    smartphone.stock -= 5
    session.commit()
    print(f"\nSmartphone actualizado: {smartphone.precio}€, stock={smartphone.stock}")

    # DELETE
    auriculares = session.query(Producto).filter_by(nombre="Auriculares").first()
    session.delete(auriculares)
    session.commit()
    print(f"\nTotal productos tras DELETE: {session.query(Producto).count()}")

---
## Cuándo usar cada herramienta

```
sqlite3 (stdlib)
  ✅ Scripts simples, herramientas internas
  ✅ Sin dependencias externas
  ✅ Solo SQLite
  ⚠️ SQL manual, más verboso

SQLAlchemy Core
  ✅ SQL expresivo en Python
  ✅ Portabilidad entre motores (SQLite, PostgreSQL, MySQL...)
  ✅ Más control que el ORM

SQLAlchemy ORM
  ✅ Aplicaciones complejas con muchas entidades relacionadas
  ✅ Código más mantenible y testeable
  ✅ Migraciones con Alembic
  ⚠️ Curva de aprendizaje inicial
  ⚠️ Puede ocultar queries ineficientes

pandas + read_sql_query
  ✅ Análisis de datos y ETL
  ✅ Integración con numpy, matplotlib, scikit-learn
```

---
## 9. Práctica — Pipeline de análisis

---

### Ejercicio 1 — CRUD completo con sqlite3

**Enunciado:** Implementa una clase `GestorTareas` que use `sqlite3` para:

- `crear(titulo, prioridad='normal')` → inserta y devuelve el id.
- `listar(prioridad=None)` → lista todas (o filtradas por prioridad).
- `completar(id)` → marca como completada.
- `eliminar(id)` → elimina la tarea.
- `resumen()` → dict con total, completadas, pendientes por prioridad.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import sqlite3
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import sqlite3
from pathlib import Path

class GestorTareas:
    def __init__(self, db_path=":memory:"):
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS tareas (
                id         INTEGER PRIMARY KEY AUTOINCREMENT,
                titulo     TEXT NOT NULL,
                completada INTEGER DEFAULT 0,
                prioridad  TEXT DEFAULT 'normal'
            )
        """)
        self.conn.commit()

    def crear(self, titulo, prioridad='normal'):
        cur = self.conn.execute(
            "INSERT INTO tareas (titulo, prioridad) VALUES (?, ?)",
            (titulo, prioridad)
        )
        self.conn.commit()
        return cur.lastrowid

    def listar(self, prioridad=None):
        if prioridad:
            cur = self.conn.execute(
                "SELECT * FROM tareas WHERE prioridad = ? ORDER BY id", (prioridad,)
            )
        else:
            cur = self.conn.execute("SELECT * FROM tareas ORDER BY prioridad DESC, id")
        return [dict(r) for r in cur.fetchall()]

    def completar(self, id_):
        self.conn.execute("UPDATE tareas SET completada = 1 WHERE id = ?", (id_,))
        self.conn.commit()

    def eliminar(self, id_):
        self.conn.execute("DELETE FROM tareas WHERE id = ?", (id_,))
        self.conn.commit()

    def resumen(self):
        cur = self.conn.execute("""
            SELECT
                COUNT(*)                AS total,
                SUM(completada)         AS completadas,
                SUM(1 - completada)     AS pendientes,
                SUM(prioridad='alta')   AS alta,
                SUM(prioridad='normal') AS normal,
                SUM(prioridad='baja')   AS baja
            FROM tareas
        """)
        return dict(cur.fetchone())


# Prueba
g = GestorTareas()
g.crear("Diseñar API", "alta")
g.crear("Escribir tests", "alta")
g.crear("Revisar docs", "baja")
g.crear("Daily standup", "normal")
g.completar(1)
g.eliminar(3)
print("Tareas:", g.listar())
print("Resumen:", g.resumen())
```
</details>

---
### Ejercicio 2 — Pipeline ETL con pandas

**Enunciado:** Implementa la función `etl_ventas(conn_tienda)` que:

1. Lea de `tienda.db` todas las líneas de pedido con sus datos relacionados (cliente, producto, categoría).
2. Calcule una columna `subtotal = cantidad × precio_ud`.
3. Genere un resumen por categoría: total vendido, unidades, importe.
4. Guarde el resumen en una nueva tabla `resumen_ventas` en la misma BD.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import sqlite3, pandas as pd
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import sqlite3, pandas as pd
from pathlib import Path

def etl_ventas(conn):
    # 1. Extraer
    df = pd.read_sql_query("""
        SELECT c.nombre  AS cliente,
               cat.nombre AS categoria,
               prod.nombre AS producto,
               pp.cantidad, pp.precio_ud
        FROM pedido_productos pp
        JOIN pedidos   p    ON pp.pedido_id   = p.id
        JOIN clientes  c    ON p.cliente_id   = c.id
        JOIN productos prod ON pp.producto_id = prod.id
        JOIN categorias cat ON prod.categoria_id = cat.id
    """, conn)

    # 2. Transformar
    df['subtotal'] = df['cantidad'] * df['precio_ud']

    # 3. Agregar
    resumen = df.groupby('categoria').agg(
        total_lineas=('subtotal', 'count'),
        unidades=('cantidad', 'sum'),
        importe=('subtotal', 'sum')
    ).round(2).reset_index()

    print("Resumen de ventas por categoría:")
    display(resumen)

    # 4. Cargar
    resumen.to_sql('resumen_ventas', conn, if_exists='replace', index=False)
    print("\nTabla 'resumen_ventas' guardada en la BD")
    return resumen

conn = sqlite3.connect(Path.home() / "tienda.db")
resumen = etl_ventas(conn)
conn.close()
```
</details>

---
### Ejercicio 3 — SQL injection

**Enunciado:** Demuestra el peligro de la interpolación de strings y cómo los parámetros lo previenen:

1. Crea una función `login_inseguro(conn, usuario, password)` con SQL interpolado.
2. Muestra cómo la entrada `' OR 1=1 --` hace que retorne todos los usuarios.
3. Implementa `login_seguro(conn, usuario, password)` con `?` y demuestra que la inyección falla.
4. Explica en comentarios por qué funciona la defensa.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import sqlite3


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import sqlite3

# BD de demo
conn = sqlite3.connect(":memory:")
conn.executescript("""
    CREATE TABLE usuarios (id INTEGER PRIMARY KEY, nombre TEXT, password TEXT);
    INSERT INTO usuarios VALUES (1, 'admin', 'secreto123');
    INSERT INTO usuarios VALUES (2, 'ana',   'pass456');
""")

# ❌ VERSIÓN INSEGURA
def login_inseguro(conn, usuario, password):
    # El SQL se construye concatenando strings: peligroso
    sql = f"SELECT * FROM usuarios WHERE nombre='{usuario}' AND password='{password}'"
    print(f"SQL ejecutado: {sql}")
    return conn.execute(sql).fetchall()

print("=== Ataque SQL injection ===")
# Entrada maliciosa: cierra la cadena y añade OR 1=1 para saltarse el filtro
resultado = login_inseguro(conn, "' OR 1=1 --", "cualquier_cosa")
print(f"Usuarios devueltos: {len(resultado)} → {resultado}")
# El atacante obtiene TODOS los usuarios sin conocer ninguna contraseña

# ✅ VERSIÓN SEGURA
def login_seguro(conn, usuario, password):
    # Los ? son marcadores de posición; sqlite3 escapa los valores automáticamente
    # Nunca se pueden usar para inyectar SQL porque NO se interpolan como código
    sql = "SELECT * FROM usuarios WHERE nombre=? AND password=?"
    return conn.execute(sql, (usuario, password)).fetchall()

print("\n=== Versión segura ===")
resultado = login_seguro(conn, "' OR 1=1 --", "cualquier_cosa")
print(f"Usuarios devueltos: {len(resultado)} → {resultado}")
# Devuelve [] porque busca literalmente un usuario llamado "' OR 1=1 --"

resultado_ok = login_seguro(conn, "admin", "secreto123")
print(f"Login correcto: {resultado_ok}")
```
</details>

---
### Ejercicio 4 — ORM básico

**Enunciado:** Con SQLAlchemy ORM, define los modelos `Libro` y `Autor` (relación 1:N). Inserta cinco autores y diez libros. Después consulta:
1. Todos los libros de un autor dado.
2. Autores que tienen más de dos libros.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, Session


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, func
from sqlalchemy.orm import declarative_base, relationship, Session

engine = create_engine("sqlite:///:memory:")
Base   = declarative_base()

class Autor(Base):
    __tablename__ = "autores"
    id     = Column(Integer, primary_key=True)
    nombre = Column(String, nullable=False)
    libros = relationship("Libro", back_populates="autor")

class Libro(Base):
    __tablename__ = "libros"
    id        = Column(Integer, primary_key=True)
    titulo    = Column(String, nullable=False)
    precio    = Column(Float)
    autor_id  = Column(Integer, ForeignKey("autores.id"))
    autor     = relationship("Autor", back_populates="libros")

Base.metadata.create_all(engine)

with Session(engine) as s:
    autores = [
        Autor(nombre="García Márquez"),
        Autor(nombre="Isabel Allende"),
        Autor(nombre="Borges"),
        Autor(nombre="Cortázar"),
        Autor(nombre="Vargas Llosa"),
    ]
    s.add_all(autores)
    s.flush()

    libros = [
        Libro(titulo="Cien años de soledad",    precio=18.5, autor=autores[0]),
        Libro(titulo="El amor en los tiempos",  precio=16.0, autor=autores[0]),
        Libro(titulo="El coronel no tiene",      precio=12.0, autor=autores[0]),
        Libro(titulo="La casa de los espíritus",precio=17.0, autor=autores[1]),
        Libro(titulo="Eva Luna",                precio=14.0, autor=autores[1]),
        Libro(titulo="Ficciones",               precio=15.0, autor=autores[2]),
        Libro(titulo="El Aleph",                precio=13.0, autor=autores[2]),
        Libro(titulo="Rayuela",                 precio=20.0, autor=autores[3]),
        Libro(titulo="Historias de Cronopios",  precio=11.0, autor=autores[3]),
        Libro(titulo="La ciudad y los perros",  precio=16.0, autor=autores[4]),
    ]
    s.add_all(libros)
    s.commit()

    # 1. Libros de García Márquez
    gm = s.query(Autor).filter_by(nombre="García Márquez").first()
    print(f"Libros de {gm.nombre}:")
    for l in gm.libros:
        print(f"  {l.titulo} — {l.precio}€")

    # 2. Autores con más de 2 libros
    print("\nAutores con > 2 libros:")
    for autor, total in (
        s.query(Autor, func.count(Libro.id).label('total'))
         .join(Libro)
         .group_by(Autor.id)
         .having(func.count(Libro.id) > 2)
         .all()
    ):
        print(f"  {autor.nombre}: {total} libros")
```
</details>

---
### Ejercicio 5 (reto) — Repositorio genérico

**Enunciado:** Implementa una clase `Repositorio` genérica con SQLAlchemy ORM que funcione con cualquier modelo:

```python
repo = Repositorio(session, Producto)
repo.crear(nombre="Teclado", precio=85.0, stock=50, categoria_id=1)
repo.obtener(1)           # por id
repo.listar()             # todos
repo.filtrar(precio=85.0) # por atributo exacto
repo.actualizar(1, precio=79.0)
repo.eliminar(1)
repo.contar()
```

Pruébalo con los modelos `Categoria` y `Producto` del demo anterior.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, Session

engine = create_engine("sqlite:///:memory:")
Base   = declarative_base()

class Categoria(Base):
    __tablename__ = "categorias"
    id      = Column(Integer, primary_key=True)
    nombre  = Column(String, nullable=False)
    productos = relationship("Producto", back_populates="categoria")
    def __repr__(self): return f"<Categoria {self.nombre}>"

class Producto(Base):
    __tablename__ = "productos"
    id           = Column(Integer, primary_key=True)
    nombre       = Column(String, nullable=False)
    precio       = Column(Float)
    stock        = Column(Integer, default=0)
    categoria_id = Column(Integer, ForeignKey("categorias.id"))
    categoria    = relationship("Categoria", back_populates="productos")
    def __repr__(self): return f"<Producto '{self.nombre}' {self.precio}€>"

Base.metadata.create_all(engine)

class Repositorio:
    def __init__(self, session, modelo):
        self.session = session
        self.modelo  = modelo

    def crear(self, **kwargs):
        obj = self.modelo(**kwargs)
        self.session.add(obj)
        self.session.commit()
        return obj

    def obtener(self, id_):
        return self.session.get(self.modelo, id_)

    def listar(self):
        return self.session.query(self.modelo).all()

    def filtrar(self, **kwargs):
        return self.session.query(self.modelo).filter_by(**kwargs).all()

    def actualizar(self, id_, **kwargs):
        obj = self.obtener(id_)
        if obj:
            for k, v in kwargs.items():
                setattr(obj, k, v)
            self.session.commit()
        return obj

    def eliminar(self, id_):
        obj = self.obtener(id_)
        if obj:
            self.session.delete(obj)
            self.session.commit()

    def contar(self):
        return self.session.query(self.modelo).count()


with Session(engine) as s:
    repo_cat  = Repositorio(s, Categoria)
    repo_prod = Repositorio(s, Producto)

    cat = repo_cat.crear(nombre="Electrónica")
    repo_prod.crear(nombre="Smartphone",  precio=599.0, stock=50, categoria_id=cat.id)
    repo_prod.crear(nombre="Auriculares", precio=89.0,  stock=100, categoria_id=cat.id)
    repo_prod.crear(nombre="Cargador",    precio=29.0,  stock=200, categoria_id=cat.id)

    print("Todos:", repo_prod.listar())
    print("Filtrado stock=100:", repo_prod.filtrar(stock=100))
    repo_prod.actualizar(1, precio=549.0)
    print("Tras actualizar:", repo_prod.obtener(1))
    repo_prod.eliminar(3)
    print("Total tras eliminar:", repo_prod.contar())
```
</details>

---
## Resumen de la clase

```python
# ── sqlite3 ────────────────────────────────────────────────────
import sqlite3
conn = sqlite3.connect("bd.db")
conn.row_factory = sqlite3.Row
cur  = conn.cursor()
cur.execute("SELECT * FROM t WHERE id = ?", (5,))   # ← parámetros SIEMPRE
cur.fetchone() · cur.fetchall() · cur.fetchmany(n)
with conn:   # transacción automática
    conn.execute("INSERT ...")

# ── pandas ─────────────────────────────────────────────────────
df = pd.read_sql_query(sql, conn, params=(val,))
df.to_sql('tabla', conn, if_exists='replace', index=False)

# ── SQLAlchemy Core ────────────────────────────────────────────
engine = create_engine("sqlite:///bd.db")
with engine.connect() as con:
    con.execute(text("SELECT ..."), {"param": valor})

# ── SQLAlchemy ORM ─────────────────────────────────────────────
class Modelo(Base):
    __tablename__ = "tabla"
    id     = Column(Integer, primary_key=True)
    nombre = Column(String)

with Session(engine) as s:
    s.add(Modelo(nombre="X"))
    s.commit()
    obj = s.get(Modelo, 1)
    objs = s.query(Modelo).filter(Modelo.nombre == "X").all()
```

## ¿Qué viene en la próxima clase?

En la **Clase 6** salimos del mundo relacional: bases de datos NoSQL con MongoDB y `pymongo`.

In [ ]:
import os
from pathlib import Path
db = Path.home() / "demo_python_sql.db"
if db.exists():
    os.remove(db)
print("Limpieza completada")